# Notebook 03 — Graphe de Connaissances Neo4j
## Module 03 · Système de Recommandation Hybride Emploi-Compétences · Cameroun
**NGOULOU-NGOUBILI Irch Defluviaire · ISE M2 · Data Science & Marketing**

---

### Objectif
Charger les 16 types de nœuds et 22 types de relations du graphe de connaissances
dans Neo4j à partir des données normalisées (Module 01) et des référentiels ESCO, MEPC, NCF.

### Plan
1. Vérification des pré-requis et connexion Neo4j
2. Création du schéma (contraintes + index)
3. Chargement ESCO (Compétences, Métiers, ISCO)
4. Chargement MEPC (3 niveaux + mapping ISCO)
5. Chargement NCF (4 niveaux)
6. Chargement Offres + nœuds contextuels
7. Chargement Candidats
8. Validation et métriques du graphe
9. Requêtes de démonstration (Skill Gap, matching, roadmap)
10. Visualisation de la structure du graphe

> **Pré-requis** : Neo4j Desktop ou Neo4j Community Edition installé et démarré
> URI : `bolt://localhost:7687` | User : `neo4j` | Password : à configurer dans `config_neo4j.py`

## 1. Vérification des pré-requis et connexion

In [1]:
import sys, json, warnings
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import importlib.util
warnings.filterwarnings('ignore')

ROOT    = Path('../').resolve()
PROC    = ROOT / 'data' / 'processed'
SRC     = ROOT / 'src' / '03_knowledge_graph'

# Import robuste de la configuration
config_path = SRC / 'config_neo4j.py'
spec = importlib.util.spec_from_file_location("config_neo4j", config_path)
config_neo4j = importlib.util.module_from_spec(spec)
spec.loader.exec_module(config_neo4j)
ESCO_DIR = config_neo4j.ESCO_DIR

sys.path.insert(0, str(SRC))

# Création du dossier PROC s'il n'existe pas
PROC.mkdir(parents=True, exist_ok=True)

# Vérification des données disponibles
print('=== DONNÉES DISPONIBLES ===')
for f in sorted(PROC.iterdir()):
    if f.suffix == '.parquet':
        df = pd.read_parquet(f)
        print(f'  {f.name:<45} {df.shape[0]:>6} lignes x {df.shape[1]} cols')


=== DONNÉES DISPONIBLES ===
  candidats_normalized.parquet                    1105 lignes x 20 cols
  mapping_isco_mepc_esco.parquet                 19987 lignes x 9 cols
  mepc_grands_groupes.parquet                        8 lignes x 5 cols
  mepc_groupes_base.parquet                        209 lignes x 7 cols
  mepc_sous_groupes.parquet                         44 lignes x 6 cols
  ncf_dom_detailles.parquet                        201 lignes x 6 cols
  ncf_dom_specialises.parquet                       31 lignes x 5 cols
  ncf_grands_domaines.parquet                       10 lignes x 4 cols
  ncf_niveaux.parquet                                9 lignes x 4 cols
  offres_normalized.parquet                       7861 lignes x 30 cols


In [2]:
# Test de connexion Neo4j
NEO4J_AVAILABLE = True  # Mettre True si Neo4j est démarré

if NEO4J_AVAILABLE:
    try:
        from neo4j import GraphDatabase
        
        driver = GraphDatabase.driver(config_neo4j.NEO4J_URI, auth=(config_neo4j.NEO4J_USER, config_neo4j.NEO4J_PASSWORD))
        driver.verify_connectivity()
        print('Connexion Neo4j OK')
        # Compter les nœuds existants
        with driver.session() as s:
            r = s.run('MATCH (n) RETURN labels(n)[0] AS label, count(n) AS n').data()
            for row in r: print(f'  {row["label"]}: {row["n"]:,}')
        driver.close()
    except Exception as e:
        print(f'Erreur de connexion Neo4j : {e}')
        print('Vérifiez la configuration dans : src/03_knowledge_graph/config_neo4j.py')
        NEO4J_AVAILABLE = False
else:
    print('Mode démonstration — Neo4j non connecté')
    print('Pour démarrer : neo4j start (ou Neo4j Desktop)')
    print('Config : src/03_knowledge_graph/config_neo4j.py')

Connexion Neo4j OK
  Compétence: 13,939
  Métier: 3,039
  GroupeISCO: 619
  GroupeCompétences: 14,579
  GrandGroupeMEPC: 8
  SousGroupeMEPC: 44
  GroupeBaseMEPC: 209
  NiveauFormationNCF: 9
  GrandDomaineNCF: 10
  DomaineSpécialiséNCF: 31
  DomaineDétailléNCF: 201


## 2. Création du schéma (contraintes + index)

Le schéma garantit :
- **Unicité** : MERGE idempotent → on peut relancer sans créer de doublons
- **Performance** : index sur les propriétés de filtrage fréquentes
- **Recherche fulltext** : index sur preferredLabel et description

In [3]:
# Afficher les contraintes du schéma
schema_path = SRC / 'schema.cypher'
cypher = schema_path.read_text(encoding='utf-8')

# Compter les statements
constraints = [s.strip() for s in cypher.split(';')
               if 'CREATE CONSTRAINT' in s]
indexes_perf = [s.strip() for s in cypher.split(';')
                if 'CREATE INDEX' in s and 'FULLTEXT' not in s]
indexes_ft   = [s.strip() for s in cypher.split(';')
                if 'FULLTEXT' in s]

print(f'Contraintes d unicite : {len(constraints)}')
print(f'Index de performance  : {len(indexes_perf)}')
print(f'Index fulltext        : {len(indexes_ft)}')
print()
print('Contraintes :')
for c in constraints[:5]:
    name_line = [l for l in c.split('\n') if 'CONSTRAINT' in l]
    if name_line: print(f'  {name_line[0].strip()[:80]}')

if NEO4J_AVAILABLE:
    from load_neo4j import create_schema, get_driver
    drv = get_driver()
    create_schema(drv)
    print('Schema cree')
    drv.close()

Contraintes d unicite : 16
Index de performance  : 13
Index fulltext        : 4

Contraintes :
  CREATE CONSTRAINT candidat_id IF NOT EXISTS
  CREATE CONSTRAINT offre_id IF NOT EXISTS
  CREATE CONSTRAINT competence_uri IF NOT EXISTS
  CREATE CONSTRAINT metier_uri IF NOT EXISTS
  CREATE CONSTRAINT groupe_comp_uri IF NOT EXISTS


11:24:10  INFO     Connexion Neo4j OK → bolt://localhost:7687
11:24:10  INFO     [1/7] Création du schéma (contraintes + index)...
11:24:10  INFO     Received notification from DBMS server: <GqlStatusObject gql_status='00NA0', status_description='note: successful completion - index or constraint already exists. `CREATE CONSTRAINT offre_id IF NOT EXISTS FOR (e:OffreEmploi) REQUIRE (e.id) IS UNIQUE` has no effect. `CONSTRAINT offre_id FOR (e:OffreEmploi) REQUIRE (e.id) IS UNIQUE` already exists.', position=None, raw_classification='SCHEMA', classification=<NotificationClassification.SCHEMA: 'SCHEMA'>, raw_severity='INFORMATION', severity=<NotificationSeverity.INFORMATION: 'INFORMATION'>, diagnostic_record={'_classification': 'SCHEMA', '_status_parameters': {'cmd': 'CREATE CONSTRAINT offre_id IF NOT EXISTS FOR (e:OffreEmploi) REQUIRE (e.id) IS UNIQUE', 'indexConstrPat': 'CONSTRAINT offre_id FOR (e:OffreEmploi) REQUIRE (e.id) IS UNIQUE'}, '_severity': 'INFORMATION', 'OPERATION': '', 'OPERA

Schema cree


## 3. Chargement ESCO — Compétences, Métiers, ISCO, Relations

**13 939 compétences** + **3 039 métiers** + **126 051 relations :NECESSITE**

La logique de chargement :
1. `MERGE` sur `conceptUri` (idempotent)
2. `SET` toutes les propriétés (mise à jour si nœud existant)
3. Enrichissement des compétences avec les collections (isDigital, isGreen, ...)
4. Enrichissement des métiers avec les codes MEPC depuis la table de mapping

In [4]:
# Simulation du chargement ESCO (sans Neo4j)
import csv

skills_data = []
with open(ESCO_DIR / 'skills_fr.csv', encoding='utf-8') as f:
    for row in csv.DictReader(f):
        skills_data.append(row)

occ_data = []
with open(ESCO_DIR / 'occupations_fr.csv', encoding='utf-8') as f:
    for row in csv.DictReader(f):
        occ_data.append(row)

rel_data = []
with open(ESCO_DIR / 'occupationSkillRelations_fr.csv', encoding='utf-8') as f:
    for row in csv.DictReader(f):
        rel_data.append(row)

print(f'Competences ESCO   : {len(skills_data):,}')
print(f'Metiers ESCO       : {len(occ_data):,}')
print(f'Relations necessite: {len(rel_data):,}')

# Distribution types de compétences
from collections import Counter
sk_types = Counter(r['skillType'] for r in skills_data)
rel_types = Counter(r['relationType'] for r in rel_data)
print(f'\nTypes competences   : {dict(sk_types)}')
print(f'Types relations     : {dict(rel_types)}')

# Nombre moyen de compétences par métier
from collections import defaultdict
skills_per_occ = defaultdict(int)
for r in rel_data:
    skills_per_occ[r['occupationUri']] += 1
counts = list(skills_per_occ.values())
print(f'\nCompetences/metier : moy={np.mean(counts):.1f}'
      f'  med={np.median(counts):.0f}  max={max(counts)}')

if NEO4J_AVAILABLE:
    from load_neo4j import load_esco_skills, load_esco_occupations, load_esco_isco, load_esco_skill_groups, load_esco_relations, get_driver
    drv = get_driver()
    load_esco_skills(drv)
    load_esco_occupations(drv)
    load_esco_isco(drv)
    load_esco_skill_groups(drv)
    load_esco_relations(drv)
    drv.close()
    print('ESCO charge dans Neo4j')

Competences ESCO   : 13,960
Metiers ESCO       : 3,043
Relations necessite: 126,051

Types competences   : {'skill/competence': 10734, 'knowledge': 3221, '': 5}
Types relations     : {'essential': 67600, 'optional': 58451}

Competences/metier : moy=41.5  med=37  max=178


11:24:27  INFO     Connexion Neo4j OK → bolt://localhost:7687
11:24:27  INFO     [2a] Chargement des Compétences ESCO...
11:24:29  INFO       [Compétences] 36%  (5,000/13,960)
11:24:31  INFO       [Compétences] 72%  (10,000/13,960)
11:24:32  INFO       [Compétences] 100%  (13,960/13,960)
11:24:32  INFO       → 13,960 compétences ESCO chargées
11:24:32  INFO     [2b] Chargement des Métiers ESCO...
11:24:35  INFO       [Métiers] 100%  (3,043/3,043)
11:24:35  INFO       → 3,043 métiers ESCO chargés
11:24:35  INFO     [2c] Chargement des GroupeISCO...
11:24:35  INFO       [GroupeISCO] 100%  (619/619)
11:24:35  INFO       → 619 groupes ISCO chargés
11:24:35  INFO     [2d] Chargement des GroupeCompétences...
11:24:36  INFO       [GroupeComp] 17%  (2,500/14,579)
11:24:36  INFO       [GroupeComp] 34%  (5,000/14,579)
11:24:36  INFO       [GroupeComp] 51%  (7,500/14,579)
11:24:36  INFO       [GroupeComp] 69%  (10,000/14,579)
11:24:36  INFO       [GroupeComp] 86%  (12,500/14,579)
11:24:36  INFO  

ESCO charge dans Neo4j


In [5]:
# Visualisation distribution compétences ESCO
NAVY, TEAL, ORANGE, GREEN = '#1E2761','#028090','#E67E22','#27AE60'

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle('Structure ESCO v1.2 — Avant chargement Neo4j', fontsize=13, fontweight='bold', color=NAVY)

# Types de skills
axes[0].pie([sk_types.get('skill/competence',0), sk_types.get('knowledge',0)],
            labels=['Compétences\n(skills)', 'Connaissances\n(knowledge)'],
            colors=[TEAL, NAVY], autopct='%1.0f%%',
            wedgeprops=dict(edgecolor='white', lw=2))
axes[0].set_title('Types de skills ESCO', fontweight='bold')

# Relations essential vs optional
axes[1].bar(['Essential', 'Optional'],
            [rel_types.get('essential',0), rel_types.get('optional',0)],
            color=[NAVY, TEAL], edgecolor='white', width=0.5)
axes[1].set_title('Relations :NECESSITE par type', fontweight='bold')
axes[1].set_ylabel('Nombre')
for i, v in enumerate([rel_types.get('essential',0), rel_types.get('optional',0)]):
    axes[1].text(i, v+500, f'{v:,}', ha='center', fontsize=10, fontweight='bold')

# Distribution skills/métier
axes[2].hist(counts, bins=30, color=ORANGE, edgecolor='white', rwidth=0.85)
axes[2].axvline(np.mean(counts), color=NAVY, lw=2, label=f'Moy={np.mean(counts):.0f}')
axes[2].set_title('Distribution skills/métier', fontweight='bold')
axes[2].set_xlabel('Nombre de compétences par métier')
axes[2].set_ylabel('Nombre de métiers')
axes[2].legend()

plt.tight_layout()
plt.savefig('fig_esco_structure.png', dpi=130, bbox_inches='tight')
plt.show()

## 4. Chargement MEPC — 3 niveaux + alignement ISCO-08

La MEPC 2013 (INS Cameroun) contextualise les métiers dans la réalité camerounaise.
L'alignement `GroupeBaseMEPC -[:ALIGNE_AVEC]-> GroupeISCO` est le **pont clé** qui
relie MEPC et ESCO via les codes ISCO-08 communs.

In [6]:
# Visualiser la structure MEPC
mepc_g = pd.read_parquet(PROC / 'mepc_grands_groupes.parquet')
mepc_s = pd.read_parquet(PROC / 'mepc_sous_groupes.parquet')
mepc_b = pd.read_parquet(PROC / 'mepc_groupes_base.parquet')
mapping = pd.read_parquet(PROC / 'mapping_isco_mepc_esco.parquet')

print('=== MEPC 2013 ===')
print(f'  Grands Groupes    : {len(mepc_g)}')
print(f'  Sous-Groupes      : {len(mepc_s)}')
print(f'  Groupes de Base   : {len(mepc_b)}')
print(f'  Total nœuds MEPC  : {len(mepc_g)+len(mepc_s)+len(mepc_b)}')

# Mapping MEPC↔ESCO
n_with_esco = mapping['esco_uri'].notna().sum()
print(f'\n=== MAPPING MEPC->ESCO ===')
print(f'  Correspondances totales  : {len(mapping):,}')
print(f'  Avec URI ESCO mappé      : {n_with_esco:,} ({n_with_esco/len(mapping)*100:.1f}%)')
print(f'  Relation Neo4j produite  : GroupeBaseMEPC -[:ALIGNE_AVEC]-> GroupeISCO')
print(f'  + Métier -[:CORRESPOND_MEPC]-> GroupeBaseMEPC')

# Grand groupe distribution
n_base_per_grand = mepc_b.groupby('code_grand_groupe').size()
print(f'\nGroupes de base par grand groupe :')
for code, n in n_base_per_grand.items():
    label = mepc_g[mepc_g['code']==str(code)]['intitule'].values
    lbl = label[0][:40] if len(label) else str(code)
    print(f'  Groupe {code} — {lbl:<40} : {n} groupes de base')

if NEO4J_AVAILABLE:
    from load_neo4j import load_mepc, get_driver
    drv = get_driver()
    load_mepc(drv)
    drv.close()

=== MEPC 2013 ===
  Grands Groupes    : 8
  Sous-Groupes      : 44
  Groupes de Base   : 209
  Total nœuds MEPC  : 261

=== MAPPING MEPC->ESCO ===
  Correspondances totales  : 19,987
  Avec URI ESCO mappé      : 19,980 (100.0%)
  Relation Neo4j produite  : GroupeBaseMEPC -[:ALIGNE_AVEC]-> GroupeISCO
  + Métier -[:CORRESPOND_MEPC]-> GroupeBaseMEPC

Groupes de base par grand groupe :
  Groupe 1 — Agriculteurs et ouvriers de l'agricultur : 18 groupes de base
  Groupe 2 — Dirigeants, directeurs, cadres de direct : 23 groupes de base
  Groupe 3 — Professions intellectuelles et scientifi : 34 groupes de base
  Groupe 4 — Professions intermédiaires               : 28 groupes de base
  Groupe 5 — Employés de type administratif           : 10 groupes de base
  Groupe 6 — Personnel des services directs aux parti : 32 groupes de base
  Groupe 7 — Artisans et ouvriers de l'industrie      : 50 groupes de base
  Groupe 8 — Forces de défense et de sécurité et pers : 14 groupes de base


11:25:25  INFO     Connexion Neo4j OK → bolt://localhost:7687
11:25:25  INFO     [3/7] Chargement des référentiels MEPC...
11:25:25  INFO       [MEPC Grands] 100%  (8/8)
11:25:25  INFO       GrandGroupeMEPC : 8
11:25:25  INFO       [MEPC Sous] 100%  (44/44)
11:25:25  INFO       SousGroupeMEPC  : 44
11:25:25  INFO       [MEPC Base] 100%  (209/209)
11:25:25  INFO       GroupeBaseMEPC  : 209
11:25:25  INFO       [MEPC GG→SG] 100%  (44/44)
11:25:26  INFO       [MEPC SG→Base] 100%  (209/209)
11:25:27  INFO       [:ALIGNE_AVEC] 100%  (203/203)
11:25:27  INFO       :ALIGNE_AVEC MEPC→ISCO : 203
11:25:29  INFO       [:CORRESPOND_MEPC] 54%  (10,000/18,642)
11:25:29  INFO       [:CORRESPOND_MEPC] 100%  (18,642/18,642)
11:25:29  INFO       :CORRESPOND_MEPC Métier→MEPC : 18642


## 5. Chargement NCF — 4 niveaux hiérarchiques

In [7]:
ncf_n = pd.read_parquet(PROC / 'ncf_niveaux.parquet')
ncf_g = pd.read_parquet(PROC / 'ncf_grands_domaines.parquet')
ncf_s = pd.read_parquet(PROC / 'ncf_dom_specialises.parquet')
ncf_d = pd.read_parquet(PROC / 'ncf_dom_detailles.parquet')

print('=== NCF 2017 ===')
for name, df in [('NiveauFormationNCF',9), ('GrandDomaineNCF',10),
                  ('DomaineSpecialiseNCF',31), ('DomaineDétailléNCF',201)]:
    print(f'  {name:<25} : {df} nœuds attendus')

print()
print('Niveaux NCF (correspondance diplômes camerounais) :')
for _, r in ncf_n.iterrows():
    print(f'  NCF-{r["code"]} — {r["intitule"]}')

if NEO4J_AVAILABLE:
    from load_neo4j import load_ncf, get_driver
    drv = get_driver()
    load_ncf(drv)
    drv.close()

=== NCF 2017 ===
  NiveauFormationNCF        : 9 nœuds attendus
  GrandDomaineNCF           : 10 nœuds attendus
  DomaineSpecialiseNCF      : 31 nœuds attendus
  DomaineDétailléNCF        : 201 nœuds attendus

Niveaux NCF (correspondance diplômes camerounais) :
  NCF-1 — Formation sur le tas
  NCF-2 — Formation dans un centre de formation (Centre de promotion de la femme et de la famille, Centre multifonctionnel, etc.)
  NCF-3 — Formation scolaire de niveau primaire et post primaire (SAR/SM, etc.)
  NCF-4 — Formation de niveau secondaire premier cycle (Centre de formation aux métiers, …)
  NCF-5 — Formation de niveau secondaire second cycle ou GCE OL (Aide-soignant, Agent technique, …)
  NCF-6 — Formation de niveau post secondaire ou supérieure à cycle court équivalant à BAC ou GCE AL +2 (BTS/HND, DUT, DSEP, DEUG, IDE, Technicien Supérieur, etc.)
  NCF-7 — Formation supérieure de premier cycle, BAC ou GCE AL + 3 (Ingénieur des travaux, Licence, …)
  NCF-8 — Formation supérieure de seco

11:25:42  INFO     Connexion Neo4j OK → bolt://localhost:7687
11:25:42  INFO     [4/7] Chargement des référentiels NCF...
11:25:42  INFO       [NCF Niveaux] 100%  (9/9)
11:25:42  INFO       NiveauFormationNCF : 9
11:25:42  INFO       [NCF GrandDom] 100%  (10/10)
11:25:42  INFO       GrandDomaineNCF    : 10
11:25:42  INFO       [NCF DomSpec] 100%  (31/31)
11:25:42  INFO       DomaineSpécialiséNCF : 31
11:25:42  INFO       [NCF DomDet] 100%  (201/201)
11:25:42  INFO       DomaineDétailléNCF   : 201
11:25:42  INFO       [NCF GD→DS] 100%  (31/31)
11:25:42  INFO       [NCF DS→DD] 100%  (201/201)
11:25:42  INFO       Hiérarchie NCF :CONTIENT créée


## 6. Chargement Offres d'emploi

7 861 offres → nœuds :OffreEmploi + nœuds contextuels (Secteur, Employeur, Localisation)

In [8]:
df_o = pd.read_parquet(PROC / 'offres_normalized.parquet')
print(f'Offres a charger : {len(df_o):,}')
print(f'Colonnes         : {list(df_o.columns)}')
print()
print('Nœuds contextuels à créer :')
print(f'  Secteurs    : {df_o["secteur_principal"].nunique()}')
print(f'  Employeurs  : {df_o["employeur"].nunique()}')
print(f'  Villes      : {df_o["ville_principale"].nunique()}')

print('\nRelations offre créées :')
print('  OffreEmploi -[:DANS_SECTEUR]-> Secteur')
print('  OffreEmploi -[:PUBLIEE_PAR]-> Employeur')
print('  OffreEmploi -[:LOCALISEE_A]-> Localisation')
print('  OffreEmploi -[:REQUIERT_NIVEAU_NCF]-> NiveauFormationNCF')
print(f'  (+ :REQUIERT -> Compétence via LLM — module 05)')

# Exemple de nœud construit
ex = df_o.iloc[0]
print('\nExemple nœud :OffreEmploi :')
for k in ['offre_id','titre_poste','employeur','ville_principale',
           'secteur_principal','type_contrat_norm','ncf_niveau_code']:
    print(f'  {k:<25} = {ex.get(k)}')

if NEO4J_AVAILABLE:
    from load_neo4j import load_offres, get_driver
    drv = get_driver()
    load_offres(drv)
    drv.close()

Offres a charger : 7,861
Colonnes         : ['offre_id', 'source', 'titre_poste', 'employeur', 'type_entreprise_norm', 'pays', 'ville_principale', 'villes_list', 'secteur_principal', 'secteurs_list', 'groupe_contrat_norm', 'type_contrat_norm', 'ncf_niveau_code', 'niveau_etudes_raw', 'experience_min_ans', 'niveau_experience_raw', 'skills_list', 'skills_raw', 'details_clean', 'details_truncated', 'details_raw', 'lien_reference', 'groupe_contrat_raw', 'type_contrat_raw', 'secteur_activite_raw', 'ville_region_raw', 'type_entreprise_raw', 'text_to_embed', 'metadata_str', 'ft_eligible']

Nœuds contextuels à créer :
  Secteurs    : 190
  Employeurs  : 956
  Villes      : 63

Relations offre créées :
  OffreEmploi -[:DANS_SECTEUR]-> Secteur
  OffreEmploi -[:PUBLIEE_PAR]-> Employeur
  OffreEmploi -[:LOCALISEE_A]-> Localisation
  OffreEmploi -[:REQUIERT_NIVEAU_NCF]-> NiveauFormationNCF
  (+ :REQUIERT -> Compétence via LLM — module 05)

Exemple nœud :OffreEmploi :
  offre_id                  = ce

11:25:54  INFO     Connexion Neo4j OK → bolt://localhost:7687
11:25:54  INFO     [5/7] Chargement des Offres d'emploi...
11:25:59  INFO       [OffreEmploi] 32%  (2,500/7,861)
11:26:01  INFO       [OffreEmploi] 64%  (5,000/7,861)
11:26:02  INFO       [OffreEmploi] 95%  (7,500/7,861)
11:26:03  INFO       [OffreEmploi] 100%  (7,861/7,861)
11:26:03  INFO       OffreEmploi : 7,861
11:26:03  INFO       [Secteurs] 100%  (190/190)
11:26:03  INFO       [Employeurs] 100%  (956/956)
11:26:03  INFO       [Localisations] 100%  (63/63)
11:26:03  INFO       Secteurs:190 Employeurs:956 Villes:63
11:26:05  INFO       [:DANS_SECTEUR] 100%  (7,861/7,861)
11:26:07  INFO       [:PUBLIEE_PAR] 100%  (7,861/7,861)
11:26:08  INFO       [:LOCALISEE_A] 100%  (7,861/7,861)
11:26:09  INFO       [:REQUIERT_NIVEAU] 100%  (7,861/7,861)
11:26:09  INFO       Relations offre contextuelles créées


## 7. Chargement Candidats

In [9]:
df_c = pd.read_parquet(PROC / 'candidats_normalized.parquet')
print(f'Candidats a charger : {len(df_c):,}')
print()
# Exemple nœud Candidat
ex = df_c.iloc[0]
print('Exemple nœud :Candidat :')
for k in ['candidat_id','metier_vise','secteur_metier','ncf_niveau_final',
           'filiere_specialite','mobilite_geo_bool','text_to_embed']:
    v = ex.get(k)
    if isinstance(v,str) and len(v)>60: v = v[:60]+'...'
    print(f'  {k:<25} = {v}')

print('\nRelations candidat créées :')
print('  Candidat -[:A_NIVEAU]-> NiveauFormationNCF')
print('  Candidat -[:A_FORMATION]-> DomaineDétailléNCF  (si filière matchée)')
print('  (+ :POSSEDE -> Compétence via déclaration — module 05)')

if NEO4J_AVAILABLE:
    from load_neo4j import load_candidats, get_driver
    drv = get_driver()
    load_candidats(drv)
    drv.close()

Candidats a charger : 1,105

Exemple nœud :Candidat :
  candidat_id               = PPKOU2501080016340
  metier_vise               = Agent de transit
  secteur_metier            = Transport, Logistique & Supply Chain
  ncf_niveau_final          = 4
  filiere_specialite        = Transport, logistique et transit
  mobilite_geo_bool         = None
  text_to_embed             = Poste: Agent de transit | Secteur: Transport, Logistique & S...

Relations candidat créées :
  Candidat -[:A_NIVEAU]-> NiveauFormationNCF
  Candidat -[:A_FORMATION]-> DomaineDétailléNCF  (si filière matchée)
  (+ :POSSEDE -> Compétence via déclaration — module 05)


11:26:26  INFO     Connexion Neo4j OK → bolt://localhost:7687
11:26:26  INFO     [6/7] Chargement des Candidats...
11:26:28  INFO       [Candidats] 100%  (1,105/1,105)
11:26:28  INFO       Candidat : 1,105
11:26:30  INFO       [:A_NIVEAU] 100%  (1,105/1,105)
11:26:30  INFO       :A_NIVEAU → 1105 relations


## 8. Validation et métriques du graphe

Résultats attendus après chargement complet.

In [ ]:
if NEO4J_AVAILABLE:
    from load_neo4j import validate_graph, get_driver
    drv = get_driver()
    stats = validate_graph(drv)
    drv.close()
    print('Validation OK')

=== RÉSULTATS ATTENDUS APRÈS CHARGEMENT COMPLET ===

Nœuds (16 types) : ~27,115
Relations attendues (base) : ~147,794

Détail nœuds :
  :Candidat                            1,105
  :OffreEmploi                         7,861
  :Compétence                         13,939
  :Métier                              3,039
  :GroupeISCO                            614
  :GroupeCompétences                      45
  :GrandGroupeMEPC                         8
  :SousGroupeMEPC                         44
  :GroupeBaseMEPC                        209
  :NiveauFormationNCF                      9
  :GrandDomaineNCF                        10
  :DomaineSpécialiséNCF                   31
  :DomaineDétailléNCF                    201
  :Secteur                          variable
  :Employeur                        variable
  :Localisation                     variable

Détail relations :
  :NECESSITE (Métier->Comp)                   126,051
  :CLASSIFIE_DANS (Métier->ISCO)                3,039
  :PLUS_LARGE_QUE 

11:32:23  INFO     Connexion Neo4j OK → bolt://localhost:7687
11:32:23  INFO     
Validation du graphe...
11:32:23  INFO     
Entité/Relation                    Compte
11:32:23  INFO     ------------------------------------------
11:32:23  INFO       Candidats                         1,105
11:32:23  INFO       OffreEmploi                       7,861
11:32:23  INFO       Compétences                      13,939
11:32:23  INFO       Métiers                           3,039
11:32:23  INFO       GroupeISCO                          619
11:32:23  INFO       GrandGroupeMEPC                       8
11:32:23  INFO       SousGroupeMEPC                       44
11:32:23  INFO       GroupeBaseMEPC                      209
11:32:23  INFO       NiveauFormationNCF                    9
11:32:23  INFO       GrandDomaineNCF                      10
11:32:23  INFO       DomaineSpécialiséNCF                 31
11:32:23  INFO       DomaineDétailléNCF                  201
11:32:23  INFO       Secteurs         

Validation OK


## 9. Requêtes de démonstration

Démonstration des requêtes Cypher utilisées dans le module 05 (GraphRAG).

In [11]:
from queries_cypher import (
    Q_SKILL_GAP_EXACT, Q_OFFRES_COMPATIBLES, Q_COMPETENCES_A_ACQUERIR,
    Q_CHEMIN_FORMATION, Q_TOP_COMPETENCES_OFFRES, Q_CANDIDATS_SIMILAIRES
)

# Afficher les requêtes clés
queries = [
    ('SKILL GAP EXACT', Q_SKILL_GAP_EXACT[:800]),
    ('OFFRES COMPATIBLES (pré-filtre)', Q_OFFRES_COMPATIBLES[:700]),
    ('COMPÉTENCES À ACQUÉRIR (roadmap)', Q_COMPETENCES_A_ACQUERIR[:600]),
]
for name, q in queries:
    print(f'=== {name} ===')
    print(q)
    print()

=== SKILL GAP EXACT ===

// Skill gap exact : compétences requises par l'offre vs possédées par le candidat
MATCH (c:Candidat   {id: $candidat_id})-[:POSSEDE]->(sc:Compétence)
MATCH (o:OffreEmploi{id: $offre_id   })-[r:REQUIERT]->(sr:Compétence)
WITH collect(DISTINCT sc.conceptUri) AS cand_skills,
     collect(DISTINCT sr.conceptUri) AS offre_skills,
     collect(DISTINCT CASE WHEN r.relationType = 'essential'
             THEN sr.conceptUri END) AS essential_skills,
     collect(DISTINCT {uri: sr.conceptUri, label: sr.preferredLabel,
             type: r.relationType}) AS offre_detail
RETURN
  [x IN offre_skills WHERE x IN cand_skills] AS acquises,
  [x IN offre_skills WHERE NOT x IN cand_skills] AS manquantes,
  [x IN essential_skills WHERE NOT x IN cand_skills] AS essentielles_manquantes,
  toFloat(size([x IN 

=== OFFRES COMPATIBLES (pré-filtre) ===

// Offres compatibles avec un candidat (filtre dur sur NCF + secteur + mobilité)
MATCH (c:Candidat {id: $candidat_id})
MATCH (o:Offre

In [12]:
# Simulation d'une requête de skill gap (sans Neo4j)
# Données simulées représentatives du graphe réel

candidat_sim = {
    'id': 'PPKOU2501080016340',
    'metier_vise': 'Agent de transit',
    'competences_possedees': [
        'utiliser des logiciels de gestion de transport',
        'communiquer avec les clients',
        'rédiger des documents administratifs',
        'gérer des fichiers et des dossiers',
    ]
}

offre_sim = {
    'id': 'offre-transit-001',
    'titre': 'Assistant Transit et Douane',
    'competences_requises': {
        'essential': [
            'utiliser des logiciels de gestion de transport',
            'appliquer les procedures douanieres',
            'coordonner les operations logistiques',
            'communiquer avec les clients',
        ],
        'optional': [
            'gerer un budget',
            'utiliser des bases de donnees',
        ]
    }
}

# Calcul du skill gap
cand_set = set(candidat_sim['competences_possedees'])
req_ess  = set(offre_sim['competences_requises']['essential'])
req_opt  = set(offre_sim['competences_requises']['optional'])
req_all  = req_ess | req_opt

acquises  = cand_set & req_all
manquantes = req_all - cand_set
ess_manquantes = req_ess - cand_set
taux = len(acquises) / len(req_all)

print('=== SIMULATION SKILL GAP ===')
print(f'Candidat   : {candidat_sim["id"]} → {candidat_sim["metier_vise"]}')
print(f'Offre      : {offre_sim["titre"]}')
print()
print(f'Taux matching : {taux:.1%} ({len(acquises)}/{len(req_all)} compétences)')
print()
print('Compétences ACQUISES :')
for c in sorted(acquises):    print(f'  ✓ {c}')
print('Compétences MANQUANTES (dont essentielles) :')
for c in sorted(manquantes):
    ess = ' [ESSENTIELLE]' if c in ess_manquantes else ''
    print(f'  ✗ {c}{ess}')

=== SIMULATION SKILL GAP ===
Candidat   : PPKOU2501080016340 → Agent de transit
Offre      : Assistant Transit et Douane

Taux matching : 33.3% (2/6 compétences)

Compétences ACQUISES :
  ✓ communiquer avec les clients
  ✓ utiliser des logiciels de gestion de transport
Compétences MANQUANTES (dont essentielles) :
  ✗ appliquer les procedures douanieres [ESSENTIELLE]
  ✗ coordonner les operations logistiques [ESSENTIELLE]
  ✗ gerer un budget
  ✗ utiliser des bases de donnees


## 10. Visualisation de la structure du graphe

In [13]:
NAVY, TEAL, ORANGE, GREEN, RED = '#1E2761','#028090','#E67E22','#27AE60','#C0392B'
PURPLE, GRAY = '#534AB7','#95A5A6'

fig, axes = plt.subplots(1, 3, figsize=(15, 6))
fig.suptitle('Structure du Graphe de Connaissances Neo4j\n'
             'Emploi-Compétences Cameroun — 16 nœuds · 22 relations',
             fontsize=13, fontweight='bold', color=NAVY)

# Donut des nœuds par famille
families = {
    'Acteurs\n(Candidat, Offre)': 7861+1105,
    'ESCO\n(Compétence, Métier)': 13939+3039+614+45,
    'MEPC\n(3 niveaux)': 8+44+209,
    'NCF\n(4 niveaux)': 9+10+31+201,
    'Contexte\n(Sect/Emp/Loc)': 400,
}
colors_f = [TEAL, NAVY, RED, GREEN, ORANGE]
wedges, texts, autos = axes[0].pie(
    list(families.values()), labels=list(families.keys()),
    autopct='%1.0f%%', colors=colors_f, startangle=90,
    wedgeprops=dict(edgecolor='white', lw=2), pctdistance=0.8)
for at in autos: at.set_fontsize(8); at.set_fontweight('bold')
axes[0].set_title('Répartition nœuds par famille', fontweight='bold')

# Barres relations
rels = {
    ':NECESSITE': 126051,
    ':PLUS_LARGE_QUE': 2488,
    ':CORRESPOND_MEPC': 19987,
    ':ALIGNE_AVEC': 400,
    ':CLASSIFIE_DANS': 3039,
    ':CONTIENT': 494,
    ':DANS_SECTEUR': 7861,
    ':PUBLIEE_PAR': 7861,
    ':A_NIVEAU': 1100,
    ':PARTIE_DE': 2500,
}
names_r = list(rels.keys())
vals_r  = list(rels.values())
colors_r= [NAVY if v>10000 else (TEAL if v>2000 else ORANGE) for v in vals_r]
y_pos = range(len(names_r))
axes[1].barh(y_pos, vals_r, color=colors_r, edgecolor='white')
axes[1].set_yticks(y_pos); axes[1].set_yticklabels(names_r, fontsize=8)
axes[1].set_xscale('log')
axes[1].set_title('Cardinalité des relations (log)', fontweight='bold')
axes[1].set_xlabel('Nombre (échelle log)')

# Heatmap connexions inter-entités
import numpy as np
entities = ['Candidat','Offre','Compétence','Métier','MEPC','NCF','ISCO']
matrix = np.zeros((7,7))
connections = [
    (0,2,1),(0,5,1),(0,1,1),  # Candidat -> Comp, NCF, Offre
    (1,2,1),(1,3,1),(1,6,1),  # Offre -> Comp, Metier, ISCO
    (3,2,1),(3,4,1),(3,6,1),  # Metier -> Comp, MEPC, ISCO
    (2,2,0.5),                # Comp -> Comp
    (4,6,0.5),(5,3,0.5),      # MEPC -> ISCO, NCF -> Metier
]
for i,j,v in connections:
    matrix[i,j] = v; matrix[j,i] = v*0.5
im = axes[2].imshow(matrix, cmap='YlOrRd', aspect='auto')
axes[2].set_xticks(range(7)); axes[2].set_yticks(range(7))
axes[2].set_xticklabels(entities, rotation=35, fontsize=8)
axes[2].set_yticklabels(entities, fontsize=8)
axes[2].set_title('Matrice de connexion inter-entités', fontweight='bold')
plt.colorbar(im, ax=axes[2], fraction=0.046)

plt.tight_layout()
plt.savefig('fig_graphe_structure.png', dpi=140, bbox_inches='tight')
plt.show()

---
## Synthèse du Module 03

| Élément | Valeur |
|---|---|
| **Nœuds chargés** | ~26 500 (16 types) |
| **Relations créées** | ~165 000 (22 types, hors :REQUIERT enrichi par LLM 05) |
| **Garantie d'idempotence** | MERGE sur clé unique → zéro doublon au rechargement |
| **Batch size** | 500 nœuds / 2000 relations par transaction |
| **Temps estimé (chargement complet)** | 5-15 min selon machine |

### Commandes

```bash
# Pipeline complet
python src/03_knowledge_graph/load_neo4j.py

# Étape par étape
python src/03_knowledge_graph/load_neo4j.py --step schema
python src/03_knowledge_graph/load_neo4j.py --step esco
python src/03_knowledge_graph/load_neo4j.py --step mepc
python src/03_knowledge_graph/load_neo4j.py --step ncf
python src/03_knowledge_graph/load_neo4j.py --step offres
python src/03_knowledge_graph/load_neo4j.py --step candidats

# Validation sans écriture
python src/03_knowledge_graph/load_neo4j.py --dry-run
```

### → Prochaine étape : Module 04 — Embeddings pgvector
Encoder toutes les entités (texte_to_embed) avec le ST fine-tuné (Module 02)
et stocker les vecteurs 384d dans PostgreSQL + pgvector avec index HNSW.